In [7]:
# ============================================================
# Functional time series simulation:
# FPCA + linear VAR baseline vs FAE + nonlinear forecast head
# ============================================================

import math
import random
from dataclasses import dataclass, asdict
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import SplineTransformer


# ============================================================
# 0) Repro
# ============================================================

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


# ============================================================
# 1) Configs
# ============================================================

@dataclass
class SimCfg:
    seed: int = 123
    T: int = 500
    P: int = 100
    d: int = 5

    # latent innovation covariance scale
    Sigma_scale: float = 0.08
    target_rho: float = 0.80

    # generator basis
    gen_basis_type: str = "bspline"   # "bspline" or "fourier"
    gen_n_basis: int = 12
    gen_bspline_degree: int = 3

    # nonlinear decoder from latent state -> basis coefficients
    map_hidden: Tuple[int, int] = (64, 64)
    map_activation: str = "relu"
    map_weight_sd: float = 0.8

    # observation noise
    meas_noise_sd: float = 0.25

    # NEW: nonlinear latent dynamics strength
    latent_nl_scale: float = 0.25
    latent_hetero_scale: float = 0.15


@dataclass
class RunCfg:
    horizon: int = 10
    fpca_K: int = 5
    fpca_var_ridge: float = 1e-6


@dataclass
class FaeCfg:
    seed: int = 743
    device: str = "cpu"

    n_basis_project: int = 30
    n_basis_revert: int = 30
    basis_type_project: str = "bspline"
    basis_type_revert: str = "bspline"
    bspline_degree: int = 3

    n_rep: int = 8   # latent representation size
    init_weight_sd: float = 0.3

    ae_epochs: int = 1200
    dyn_epochs: int = 800

    batch_size: int = 32
    lr_ae: float = 3e-4
    lr_dyn: float = 3e-4
    weight_decay: float = 1e-4

    split_rate: float = 0.85
    log_every: int = 100

    # optional coefficient smoothness penalty
    lamb: float = 1e-4

    # multi-step rollout training
    dyn_teacher_forcing_noise: float = 0.01


# ============================================================
# 2) Small utilities
# ============================================================

def _mse(a, b) -> float:
    a = torch.as_tensor(a, dtype=torch.float32)
    b = torch.as_tensor(b, dtype=torch.float32)
    return float(torch.mean((a - b) ** 2).item())

def rel_mse(pred, truth) -> float:
    pred = torch.as_tensor(pred, dtype=torch.float32)
    truth = torch.as_tensor(truth, dtype=torch.float32)
    num = torch.mean((pred - truth) ** 2)
    den = torch.mean(truth ** 2) + 1e-12
    return float((num / den).item())

def chrono_split(X: torch.Tensor, horizon: int) -> Tuple[torch.Tensor, torch.Tensor]:
    return X[:-horizon], X[-horizon:]

def activation_from_name(name: str) -> nn.Module:
    name = name.lower()
    if name == "relu":
        return nn.ReLU()
    if name == "tanh":
        return nn.Tanh()
    if name == "gelu":
        return nn.GELU()
    raise ValueError(f"Unknown activation: {name}")


# ============================================================
# 3) Basis builders
# ============================================================

def make_grid(P: int) -> np.ndarray:
    return np.linspace(0.0, 1.0, P)

def make_bspline_basis(u: np.ndarray, n_basis: int, degree: int) -> np.ndarray:
    # sklearn SplineTransformer returns basis matrix [P, n_features]
    st = SplineTransformer(
        n_knots=max(n_basis - degree + 1, degree + 1),
        degree=degree,
        include_bias=True
    )
    B = st.fit_transform(u.reshape(-1, 1))
    if B.shape[1] > n_basis:
        B = B[:, :n_basis]
    elif B.shape[1] < n_basis:
        pad = np.zeros((B.shape[0], n_basis - B.shape[1]))
        B = np.hstack([B, pad])
    return B.astype(np.float32)

def make_fourier_basis(u: np.ndarray, n_basis: int) -> np.ndarray:
    cols = [np.ones_like(u)]
    k = 1
    while len(cols) < n_basis:
        cols.append(np.sin(2 * np.pi * k * u))
        if len(cols) < n_basis:
            cols.append(np.cos(2 * np.pi * k * u))
        k += 1
    B = np.column_stack(cols[:n_basis]).astype(np.float32)
    return B

def make_basis(u: np.ndarray, basis_type: str, n_basis: int, degree: int = 3) -> np.ndarray:
    basis_type = basis_type.lower()
    if basis_type == "bspline":
        return make_bspline_basis(u, n_basis, degree)
    if basis_type == "fourier":
        return make_fourier_basis(u, n_basis)
    raise ValueError(f"Unknown basis_type: {basis_type}")


# ============================================================
# 4) Generator pieces
# ============================================================

class MLP(nn.Module):
    def __init__(self, in_dim: int, hidden: Tuple[int, ...], out_dim: int, act: str = "relu"):
        super().__init__()
        layers: List[nn.Module] = []
        prev = in_dim
        for h in hidden:
            layers.append(nn.Linear(prev, h))
            layers.append(activation_from_name(act))
            prev = h
        layers.append(nn.Linear(prev, out_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


def stabilize_A(A: np.ndarray, target_rho: float = 0.8) -> np.ndarray:
    eigvals = np.linalg.eigvals(A)
    rho = float(np.max(np.abs(eigvals)))
    if rho < 1e-12:
        return A
    return A * (target_rho / rho)


def generate_latent_nonlinear_ar1(
    T: int,
    d: int,
    Sigma: np.ndarray,
    seed: int,
    target_rho: float = 0.8,
    nl_scale: float = 0.25,
    hetero_scale: float = 0.15,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Mildly nonlinear latent dynamics.
    Still close to AR(1), but no longer ideal for linear FPCA forecasting.

    z_t = A z_{t-1}
          + nonlinear_drift(z_{t-1})
          + state-dependent noise
    """
    rng = np.random.default_rng(seed)

    A = rng.normal(size=(d, d)) * 0.20
    A = stabilize_A(A, target_rho=target_rho)

    Z = np.zeros((T, d), dtype=np.float32)
    Z[0] = rng.multivariate_normal(np.zeros(d), Sigma).astype(np.float32)

    for t in range(1, T):
        zprev = Z[t - 1].copy()
        Az = A @ zprev

        # Local nonlinear drift on first coordinates
        drift = np.zeros(d, dtype=np.float32)
        if d >= 1:
            drift[0] += nl_scale * (zprev[0] ** 2 - 0.75)
        if d >= 2:
            drift[1] += 0.8 * nl_scale * np.sin(1.5 * zprev[1])
        if d >= 3:
            drift[2] += 0.6 * nl_scale * np.tanh(zprev[0] * zprev[2])
        if d >= 4:
            drift[3] += 0.5 * nl_scale * (zprev[1] * zprev[3])

        scale = 1.0 + hetero_scale * abs(float(zprev[0]))
        eps = rng.multivariate_normal(np.zeros(d), (scale ** 2) * Sigma).astype(np.float32)

        Z[t] = Az + drift + eps

    return Z, A


def build_fixed_decoder(sim_cfg: SimCfg) -> Dict[str, object]:
    set_seed(sim_cfg.seed)

    u = make_grid(sim_cfg.P)
    Bgen_np = make_basis(
        u=u,
        basis_type=sim_cfg.gen_basis_type,
        n_basis=sim_cfg.gen_n_basis,
        degree=sim_cfg.gen_bspline_degree
    )

    mapper = MLP(
        in_dim=sim_cfg.d,
        hidden=sim_cfg.map_hidden,
        out_dim=sim_cfg.gen_n_basis,
        act=sim_cfg.map_activation,
    )

    for m in mapper.modules():
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=sim_cfg.map_weight_sd / math.sqrt(m.in_features))
            nn.init.zeros_(m.bias)

    return {
        "u": torch.tensor(u, dtype=torch.float32),
        "tpts": torch.tensor(u, dtype=torch.float32),
        "Bgen": torch.tensor(Bgen_np, dtype=torch.float32),
        "mapper": mapper.eval(),
    }


def simulate_functional_ts(sim_cfg: SimCfg) -> Dict[str, torch.Tensor]:
    fixed = build_fixed_decoder(sim_cfg)
    return _simulate_new_dataset_same_decoder(fixed, sim_cfg, data_seed=sim_cfg.seed + 1000)


def _simulate_new_dataset_same_decoder(
    fixed: Dict[str, object],
    sim_cfg: SimCfg,
    data_seed: int,
) -> Dict[str, torch.Tensor]:
    Sigma = (sim_cfg.Sigma_scale ** 2) * np.eye(sim_cfg.d, dtype=np.float32)

    Z, A_true = generate_latent_nonlinear_ar1(
        T=sim_cfg.T,
        d=sim_cfg.d,
        Sigma=Sigma,
        seed=data_seed,
        target_rho=sim_cfg.target_rho,
        nl_scale=sim_cfg.latent_nl_scale,
        hetero_scale=sim_cfg.latent_hetero_scale,
    )

    Z_t = torch.tensor(Z, dtype=torch.float32)

    mapper: nn.Module = fixed["mapper"]
    Bgen: torch.Tensor = fixed["Bgen"]

    with torch.no_grad():
        coef = mapper(Z_t)                   # [T, M]
        X_clean = coef @ Bgen.T              # [T, P]
        X_noisy = X_clean + sim_cfg.meas_noise_sd * torch.randn_like(X_clean)

    return {
        "u": fixed["u"],
        "tpts": fixed["tpts"],
        "Z": Z_t,
        "A_true": torch.tensor(A_true, dtype=torch.float32),
        "X_clean": X_clean,
        "X_noisy": X_noisy,
    }


# ============================================================
# 5) FPCA baseline
# ============================================================

def fit_var1(Y: np.ndarray, ridge: float = 1e-6) -> Tuple[np.ndarray, np.ndarray]:
    """
    Y_t = b + A Y_{t-1}
    """
    X = Y[:-1]
    Z = Y[1:]
    X1 = np.hstack([np.ones((X.shape[0], 1)), X])  # [n, 1+p]

    XtX = X1.T @ X1
    reg = ridge * np.eye(X1.shape[1], dtype=np.float64)
    B = np.linalg.solve(XtX + reg, X1.T @ Z)       # [1+p, p]

    b = B[0]
    A = B[1:].T
    return b.astype(np.float32), A.astype(np.float32)

def forecast_var1(y_last: np.ndarray, b: np.ndarray, A: np.ndarray, steps: int) -> np.ndarray:
    out = []
    cur = y_last.astype(np.float32).copy()
    for _ in range(steps):
        cur = b + A @ cur
        out.append(cur.copy())
    return np.stack(out, axis=0)

def fpca_reconstruct_uncentered_from_trainfit(
    X_train: torch.Tensor,
    X_eval: torch.Tensor,
    K: int,
) -> Tuple[torch.Tensor, PCA, np.ndarray]:
    X_mean = X_train.mean(dim=0)
    X_train_c = X_train - X_mean
    X_eval_c = X_eval - X_mean

    pca = PCA(n_components=K, svd_solver="full")
    scores_train = pca.fit_transform(X_train_c.numpy())
    scores_eval = pca.transform(X_eval_c.numpy())

    Xhat = pca.inverse_transform(scores_eval)
    Xhat = Xhat + X_mean.numpy()
    return torch.tensor(Xhat, dtype=torch.float32), pca, scores_train

def fpca_var_forecast_uncentered(
    X_train: torch.Tensor,
    K: int,
    steps: int,
    var_ridge: float = 1e-6
) -> torch.Tensor:
    X_mean = X_train.mean(dim=0)
    X_train_c = X_train - X_mean

    pca = PCA(n_components=K, svd_solver="full")
    scores = pca.fit_transform(X_train_c.numpy())

    b, A = fit_var1(scores, ridge=var_ridge)
    Sf = forecast_var1(scores[-1], b, A, steps)

    Xf = pca.inverse_transform(Sf)
    Xf = Xf + X_mean.numpy()
    return torch.tensor(Xf, dtype=torch.float32)


# ============================================================
# 6) FAE model
# ============================================================

class FAEVanilla(nn.Module):
    """
    Project x(t) onto basis Bp -> coefficients -> encoder -> latent rep
    latent rep -> decoder -> coefficients on Br -> reconstruct curve
    """
    def __init__(
        self,
        P: int,
        n_basis_project: int,
        n_basis_revert: int,
        n_rep: int,
        hidden: Tuple[int, int] = (64, 64),
    ):
        super().__init__()

        self.enc = nn.Sequential(
            nn.Linear(n_basis_project, hidden[0]),
            nn.ReLU(),
            nn.Linear(hidden[0], hidden[1]),
            nn.ReLU(),
            nn.Linear(hidden[1], n_rep),
        )

        self.dec = nn.Sequential(
            nn.Linear(n_rep, hidden[1]),
            nn.ReLU(),
            nn.Linear(hidden[1], hidden[0]),
            nn.ReLU(),
            nn.Linear(hidden[0], n_basis_revert),
        )

        self.P = P
        self.n_basis_project = n_basis_project
        self.n_basis_revert = n_basis_revert
        self.n_rep = n_rep

    def encode_from_curve(self, X: torch.Tensor, Bp: torch.Tensor) -> torch.Tensor:
        # least-squares projection onto basis
        # coef = argmin ||X - coef Bp^T||^2
        G = Bp.T @ Bp + 1e-6 * torch.eye(Bp.shape[1], device=Bp.device)
        coef = torch.linalg.solve(G, Bp.T @ X.T).T
        H = self.enc(coef)
        return H

    def decode_from_rep(self, H: torch.Tensor, Br: torch.Tensor) -> torch.Tensor:
        coef = self.dec(H)
        Xhat = coef @ Br.T
        return Xhat

    def forward(self, X: torch.Tensor, Bp: torch.Tensor, Br: torch.Tensor):
        H = self.encode_from_curve(X, Bp)
        coef_r = self.dec(H)
        Xhat = coef_r @ Br.T
        return Xhat, H, coef_r


class LatentDynamicsMLP(nn.Module):
    """
    One-step latent transition: h_{t+1} = g(h_t)
    """
    def __init__(self, n_rep: int, hidden: Tuple[int, int] = (64, 64)):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_rep, hidden[0]),
            nn.ReLU(),
            nn.Linear(hidden[0], hidden[1]),
            nn.ReLU(),
            nn.Linear(hidden[1], n_rep),
        )

    def forward(self, h):
        return self.net(h)


def diff_penalty(coef: torch.Tensor) -> torch.Tensor:
    if coef.shape[1] <= 1:
        return torch.tensor(0.0, device=coef.device)
    d1 = coef[:, 1:] - coef[:, :-1]
    return torch.mean(d1 ** 2)


def init_small_weights(model: nn.Module, sd: float) -> None:
    for m in model.modules():
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=sd / math.sqrt(m.in_features))
            nn.init.zeros_(m.bias)


def train_fae_on_noisy_train(
    Xn_train: torch.Tensor,
    tpts: torch.Tensor,
    cfg: FaeCfg,
) -> Tuple[FAEVanilla, torch.Tensor, torch.Tensor, torch.Tensor]:
    set_seed(cfg.seed)
    device = torch.device(cfg.device)

    P = Xn_train.shape[1]
    u = tpts.numpy()

    Bp_np = make_basis(u, cfg.basis_type_project, cfg.n_basis_project, cfg.bspline_degree)
    Br_np = make_basis(u, cfg.basis_type_revert, cfg.n_basis_revert, cfg.bspline_degree)

    Bp = torch.tensor(Bp_np, dtype=torch.float32, device=device)
    Br = torch.tensor(Br_np, dtype=torch.float32, device=device)

    model = FAEVanilla(
        P=P,
        n_basis_project=cfg.n_basis_project,
        n_basis_revert=cfg.n_basis_revert,
        n_rep=cfg.n_rep,
        hidden=(64, 64),
    ).to(device)

    init_small_weights(model, cfg.init_weight_sd)

    idx_all = np.arange(Xn_train.shape[0])
    idx_tr, idx_va = train_test_split(
        idx_all,
        train_size=cfg.split_rate,
        random_state=cfg.seed,
        shuffle=True
    )

    Xtr = Xn_train[idx_tr].to(device).float()
    Xva = Xn_train[idx_va].to(device).float()

    opt = optim.Adam(model.parameters(), lr=cfg.lr_ae, weight_decay=cfg.weight_decay)
    loss_fn = nn.MSELoss()

    for ep in range(1, cfg.ae_epochs + 1):
        model.train()
        perm = torch.randperm(Xtr.shape[0], device=device)

        for i in range(0, Xtr.shape[0], cfg.batch_size):
            batch = Xtr[perm[i:i + cfg.batch_size]]
            opt.zero_grad()
            xhat, _, coef = model(batch, Bp, Br)
            loss = loss_fn(xhat, batch) + cfg.lamb * diff_penalty(coef)
            loss.backward()
            opt.step()

        if cfg.log_every and (ep % cfg.log_every == 0):
            model.eval()
            with torch.no_grad():
                tr_hat, _, _ = model(Xtr, Bp, Br)
                va_hat, _, _ = model(Xva, Bp, Br)
                tr = loss_fn(tr_hat, Xtr).item()
                va = loss_fn(va_hat, Xva).item()
            print(f"[AE ] ep {ep:4d} | train_mse={tr:.6e} | val_mse={va:.6e}")

    return model.eval(), tpts.detach().cpu(), Bp.detach().cpu(), Br.detach().cpu()


@torch.no_grad()
def fae_reconstruct(
    model: FAEVanilla,
    X: torch.Tensor,
    Bp: torch.Tensor,
    Br: torch.Tensor
) -> Tuple[torch.Tensor, torch.Tensor]:
    device = next(model.parameters()).device
    X = X.to(device).float()
    Bp = Bp.to(device)
    Br = Br.to(device)
    Xhat, H, _ = model(X, Bp, Br)
    return Xhat.detach().cpu(), H.detach().cpu()


def train_fae_dynamics_head(
    model: FAEVanilla,
    Xn_train: torch.Tensor,
    Bp: torch.Tensor,
    cfg: FaeCfg,
) -> LatentDynamicsMLP:
    """
    Train nonlinear one-step latent transition on encoded FAE states.
    """
    set_seed(cfg.seed + 17)
    device = next(model.parameters()).device
    Bp = Bp.to(device)

    model.eval()
    with torch.no_grad():
        H = model.encode_from_curve(Xn_train.to(device).float(), Bp)

    H_in = H[:-1]
    H_out = H[1:]

    dyn = LatentDynamicsMLP(n_rep=cfg.n_rep, hidden=(64, 64)).to(device)
    init_small_weights(dyn, cfg.init_weight_sd)

    opt = optim.Adam(dyn.parameters(), lr=cfg.lr_dyn, weight_decay=cfg.weight_decay)
    loss_fn = nn.MSELoss()

    for ep in range(1, cfg.dyn_epochs + 1):
        dyn.train()
        perm = torch.randperm(H_in.shape[0], device=device)

        for i in range(0, H_in.shape[0], cfg.batch_size):
            idx = perm[i:i + cfg.batch_size]
            hin = H_in[idx]
            hout = H_out[idx]

            if cfg.dyn_teacher_forcing_noise > 0:
                hin = hin + cfg.dyn_teacher_forcing_noise * torch.randn_like(hin)

            pred = dyn(hin)
            loss = loss_fn(pred, hout)

            opt.zero_grad()
            loss.backward()
            opt.step()

        if cfg.log_every and (ep % cfg.log_every == 0):
            dyn.eval()
            with torch.no_grad():
                pred = dyn(H_in)
                tr = loss_fn(pred, H_out).item()
            print(f"[DYN] ep {ep:4d} | one_step_latent_mse={tr:.6e}")

    return dyn.eval()


@torch.no_grad()
def fae_nonlinear_forecast(
    model: FAEVanilla,
    dyn: LatentDynamicsMLP,
    Xn_train: torch.Tensor,
    steps: int,
    Bp: torch.Tensor,
    Br: torch.Tensor,
) -> torch.Tensor:
    device = next(model.parameters()).device
    Bp = Bp.to(device)
    Br = Br.to(device)

    H = model.encode_from_curve(Xn_train.to(device).float(), Bp)
    h = H[-1:].clone()

    preds = []
    for _ in range(steps):
        h = dyn(h)
        xhat = model.decode_from_rep(h, Br)
        preds.append(xhat.squeeze(0).detach().cpu())

    return torch.stack(preds, dim=0)


# ============================================================
# 7) One run
# ============================================================

def run_one_rep(
    fixed: Dict[str, object],
    sim_cfg: SimCfg,
    run_cfg: RunCfg,
    fae_cfg: FaeCfg,
    rep: int,
    data_seed_base: int,
    verbose: bool = False,
) -> Dict[str, float]:
    data_seed = data_seed_base + rep
    sim = _simulate_new_dataset_same_decoder(fixed, sim_cfg, data_seed=data_seed)

    tpts = sim["tpts"]
    X_clean = sim["X_clean"]
    X_noisy = sim["X_noisy"]

    Xc_train, Xc_test = chrono_split(X_clean, run_cfg.horizon)
    Xn_train, Xn_test = chrono_split(X_noisy, run_cfg.horizon)

    # ---------- FPCA reconstruction ----------
    X_fpca_recon_train, _, _ = fpca_reconstruct_uncentered_from_trainfit(
        X_train=Xn_train,
        X_eval=Xn_train,
        K=run_cfg.fpca_K,
    )

    # ---------- FAE reconstruction ----------
    old_log = fae_cfg.log_every
    if not verbose:
        fae_cfg.log_every = 0

    model_fae, tpts_fae, Bp_fae, Br_fae = train_fae_on_noisy_train(Xn_train, tpts, fae_cfg)
    Bp_dev = Bp_fae.to(fae_cfg.device)
    Br_dev = Br_fae.to(fae_cfg.device)

    X_fae_recon_train, _ = fae_reconstruct(model_fae, Xn_train, Bp_dev, Br_dev)

    # ---------- Forecast ----------
    X_fpca_fore = fpca_var_forecast_uncentered(
        X_train=Xn_train,
        K=run_cfg.fpca_K,
        steps=run_cfg.horizon,
        var_ridge=run_cfg.fpca_var_ridge,
    )

    dyn_head = train_fae_dynamics_head(model_fae, Xn_train, Bp_dev, fae_cfg)
    X_fae_fore = fae_nonlinear_forecast(
        model=model_fae,
        dyn=dyn_head,
        Xn_train=Xn_train,
        steps=run_cfg.horizon,
        Bp=Bp_dev,
        Br=Br_dev,
    )

    fae_cfg.log_every = old_log

    # ---------- Metrics ----------
    # Reconstruction
    fpca_recon_rel_clean = rel_mse(X_fpca_recon_train, Xc_train)
    fpca_recon_rel_noisy = rel_mse(X_fpca_recon_train, Xn_train)
    fae_recon_rel_clean = rel_mse(X_fae_recon_train, Xc_train)
    fae_recon_rel_noisy = rel_mse(X_fae_recon_train, Xn_train)

    # Forecast
    fpca_fore_rel_clean = rel_mse(X_fpca_fore, Xc_test)
    fpca_fore_rel_noisy = rel_mse(X_fpca_fore, Xn_test)
    fae_fore_rel_clean = rel_mse(X_fae_fore, Xc_test)
    fae_fore_rel_noisy = rel_mse(X_fae_fore, Xn_test)

    # Final AE train/val MSE
    idx_all = np.arange(Xn_train.shape[0])
    idx_tr, idx_va = train_test_split(
        idx_all,
        train_size=fae_cfg.split_rate,
        random_state=fae_cfg.seed,
        shuffle=True
    )
    Xtr = Xn_train[idx_tr].float()
    Xva = Xn_train[idx_va].float()

    model_fae.eval()
    with torch.no_grad():
        xtr_hat, _, _ = model_fae(Xtr.to(fae_cfg.device), Bp_dev, Br_dev)
        xva_hat, _, _ = model_fae(Xva.to(fae_cfg.device), Bp_dev, Br_dev)

    row = {
        "rep": rep,
        "data_seed": data_seed,

        "fpca_recon_relMSE_vs_clean": fpca_recon_rel_clean,
        "fpca_recon_relMSE_vs_noisy": fpca_recon_rel_noisy,
        "fae_recon_relMSE_vs_clean": fae_recon_rel_clean,
        "fae_recon_relMSE_vs_noisy": fae_recon_rel_noisy,

        "fae_train_mse_final_noisy": _mse(xtr_hat.detach().cpu(), Xtr),
        "fae_val_mse_final_noisy": _mse(xva_hat.detach().cpu(), Xva),

        "fpca_fore_relMSE_vs_clean": fpca_fore_rel_clean,
        "fpca_fore_relMSE_vs_noisy": fpca_fore_rel_noisy,
        "fae_fore_relMSE_vs_clean": fae_fore_rel_clean,
        "fae_fore_relMSE_vs_noisy": fae_fore_rel_noisy,
    }

    row.update({
        "map_mode": "nonlinear",
        "gen_basis_type": sim_cfg.gen_basis_type,
        "gen_n_basis": sim_cfg.gen_n_basis,
        "T": sim_cfg.T,
        "P": sim_cfg.P,
        "d": sim_cfg.d,
        "meas_noise_sd": sim_cfg.meas_noise_sd,
        "fpca_K": run_cfg.fpca_K,
        "horizon": run_cfg.horizon,
        "fae_n_rep": fae_cfg.n_rep,
        "fae_n_basis_project": fae_cfg.n_basis_project,
        "fae_n_basis_revert": fae_cfg.n_basis_revert,
        "latent_nl_scale": sim_cfg.latent_nl_scale,
        "latent_hetero_scale": sim_cfg.latent_hetero_scale,
    })
    return row


# ============================================================
# 8) Many runs
# ============================================================

def run_sim_fpca_fae_many(
    sim_cfg: SimCfg,
    run_cfg: RunCfg,
    fae_cfg: FaeCfg,
    n_reps: int = 10,
    data_seed_base: int = 30000,
    verbose: bool = False,
) -> pd.DataFrame:
    fixed = build_fixed_decoder(sim_cfg)

    rows: List[Dict[str, float]] = []
    for rep in range(1, n_reps + 1):
        print(f"=== repetition {rep}/{n_reps} ===")
        row = run_one_rep(
            fixed=fixed,
            sim_cfg=sim_cfg,
            run_cfg=run_cfg,
            fae_cfg=fae_cfg,
            rep=rep,
            data_seed_base=data_seed_base,
            verbose=verbose,
        )
        rows.append(row)

        print(
            f"recon(clean): FPCA={row['fpca_recon_relMSE_vs_clean']:.4f} | "
            f"FAE={row['fae_recon_relMSE_vs_clean']:.4f}"
        )
        print(
            f"fore(clean) : FPCA={row['fpca_fore_relMSE_vs_clean']:.4f} | "
            f"FAE={row['fae_fore_relMSE_vs_clean']:.4f}"
        )
        print()

    df = pd.DataFrame(rows)
    return df


# ============================================================
# 9) Example
# ============================================================

if __name__ == "__main__":
    sim_cfg = SimCfg(
        seed=123,
        T=1000,
        P=100,
        d=5,
        Sigma_scale=0.08,
        target_rho=0.80,
        gen_basis_type="bspline",
        gen_n_basis=12,
        gen_bspline_degree=3,
        map_hidden=(64, 64),
        map_activation="relu",
        map_weight_sd=0.8,
        meas_noise_sd=0.10,      # smaller than before
        latent_nl_scale=0.25,    # NEW
        latent_hetero_scale=0.15 # NEW
    )

    run_cfg = RunCfg(
        horizon=10,              # longer horizon helps expose nonlinear forecast gains
        fpca_K=8,
        fpca_var_ridge=1e-3,
    )

    fae_cfg = FaeCfg(
        seed=743,
        device="cpu",
        n_basis_project=30,
        n_basis_revert=30,
        basis_type_project="bspline",
        basis_type_revert="bspline",
        bspline_degree=3,
        n_rep=8,                 # slightly richer latent space than 5
        init_weight_sd=0.3,
        ae_epochs=1200,
        dyn_epochs=800,
        batch_size=32,
        lr_ae=3e-4,
        lr_dyn=3e-4,
        weight_decay=1e-4,
        split_rate=0.85,
        log_every=200,
        lamb=1e-4,
        dyn_teacher_forcing_noise=0.01,
    )

    df = run_sim_fpca_fae_many(
        sim_cfg=sim_cfg,
        run_cfg=run_cfg,
        fae_cfg=fae_cfg,
        n_reps=10,
        data_seed_base=30000,
        verbose=False,
    )

    print("\n========== averages over repetitions ==========")
    avg_cols = [
        "fpca_recon_relMSE_vs_clean",
        "fae_recon_relMSE_vs_clean",
        "fpca_fore_relMSE_vs_clean",
        "fae_fore_relMSE_vs_clean",
        "fpca_fore_relMSE_vs_noisy",
        "fae_fore_relMSE_vs_noisy",
    ]
    print(df[avg_cols].mean())

    print("\n========== full table ==========")
    print(df.head(10))


df.to_excel("sim_results_10runs_MLP_new3.xlsx", index=False)

=== repetition 1/10 ===
recon(clean): FPCA=0.0588 | FAE=0.0104
fore(clean) : FPCA=0.0155 | FAE=0.0090

=== repetition 2/10 ===
recon(clean): FPCA=0.8070 | FAE=0.4110
fore(clean) : FPCA=0.4930 | FAE=0.5174

=== repetition 3/10 ===
recon(clean): FPCA=1.1726 | FAE=0.3112
fore(clean) : FPCA=0.4846 | FAE=0.4723

=== repetition 4/10 ===
recon(clean): FPCA=1.0591 | FAE=0.4467
fore(clean) : FPCA=0.5145 | FAE=0.6000

=== repetition 5/10 ===
recon(clean): FPCA=2.1166 | FAE=0.4890
fore(clean) : FPCA=0.5056 | FAE=0.4960

=== repetition 6/10 ===
recon(clean): FPCA=1.0722 | FAE=0.3402
fore(clean) : FPCA=0.4320 | FAE=0.4382

=== repetition 7/10 ===
recon(clean): FPCA=1.4796 | FAE=0.4383
fore(clean) : FPCA=0.5622 | FAE=0.5403

=== repetition 8/10 ===
recon(clean): FPCA=0.0432 | FAE=0.0072
fore(clean) : FPCA=0.0126 | FAE=0.0095

=== repetition 9/10 ===
recon(clean): FPCA=0.4006 | FAE=0.0902
fore(clean) : FPCA=0.2991 | FAE=0.2960

=== repetition 10/10 ===
recon(clean): FPCA=1.1809 | FAE=0.2584
fore(clea